# 1. The problem: the model lives in a notebook. The plan for this module

*Model deployment* is crucial when you need to use the model across different machines or applications without having to retrain or rerun the code. 

By deploying the model as a web service, external systems (like marketing services) can send requests to the server to get predictions, such as whether a customer is likely to churn. Based on the prediction, actions like sending promotional offers can be automated.

To deploy the model in a server there are some steps:

1. Train and save the model: after training the model, save it as a file, to use it for making predictions in future (session 02-pickle).
2. Create API endpoints: make the API endpoints in order to request predictions. It is possible to use the Flask framework to create web service API endpoints that other services can interact with (session 03-flask-intro and 04-flask-deployment).
3. Some other server deployment options (sessions 5 to 9):
    * Pipenv: Create isolated environments to manage the Python dependencies of the web service, ensuring they don't interfere with other services on the machine.
    * Docker: Package the service in a Docker container, which includes both system and Python dependencies, making it easier to deploy consistently across different environments.
    * Deploy to the Cloud: Finally, deploy the Docker container to a cloud service like AWS to make the model accessible globally, ensuring scalability and reliability.

# 2. Saving and loading the model

Training gives us two objects: the DictVectorizer that turns customer records into a feature matrix, and the logistic regression model itself. Both live in the memory of the notebook process - close the notebook and they are gone.

Re-training the model every time we want a prediction is wasteful: training takes time, and we would get a slightly different model depending on the data split and random state. What we want instead is to train once, save the result to a file, and later simply load that file in the web service.

## Saving with pickle

*Pickle* is a standard Python library for serializing objects: it writes a Python object into a binary file, and can read it back later. 

We save both the vectorizer and the model together as a tuple:

In [ ]:
import pickle

output_file = f'model_C={C}.bin'

with open(output_file, 'wb') as f_out:
    pickle.dump((dv, model), f_out)

* The `wb`mode means write-binary: pickle produces binary data, so we must not open the file in text mode.
* We name the file with the value of `C`we used, because after tuning the previous module we know this parameter, and it is useful to see it in the filename: `model_C=1.0.bin`
* The vectorizer has to travel with the model. It was fitted on the training data and it "remembers" which categorical values map to which feature columns.

## Loading the model

To use the model - for example, from a web service - we load the file back:

In [ ]:
with open('model_C=1.0.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

* Here `rb` is read-binary. 
* `pickle.load` returns the tuple we saved, and we unpack it into `dv` and `model`. 
* From this point we can score customers without running any training code.

# 3. Web services: introduction to Flask

## A ping/pong service in Flask

In [ ]:
from flask import Flask

app = Flask('ping')                                         #creates the application and gives it a name

@app.route('/ping', methods=['GET'])                        #it tells Flask that the function below should handle GET requests to the /ping address
def ping():
    return "PONG"

if __name__ == "__main__":                                  #block runs the development server, but only when we execute the file directly (python ping.py)
    app.run(debug=True, host='0.0.0.0', port=9696)          # host='0.0.0.0' makes the server listen on all network interfaces (not only localhost), port=9696 sets the port

A note on addresses: localhost (also 127.0.0.1) is the machine itself. Binding to 0.0.0.0 instead means "accept requests coming from the network", which is what we want once the service runs on a server. For local development either works.

## Testing it

In [ ]:
python3 ping.py

Then, from another terminal, query it with curl - a command-line tool for making HTTP requests:

In [ ]:
curl http://localhost:9696/ping

Opening http://localhost:9696/ping in a browser works as well - the browser also sends a GET request. With that, we have a running web service.

# 4. Serving the churn model with Flask

## The web service

Guarda predict.py:

* The model is loaded once, at start-up - not on every request. Loading is fast compared to training, but there is no reason to repeat it.
* The route uses the POST method: the customer data comes in the body of the request as JSON, and request.get_json() turns it into a Python dictionary.
* We transform the customer with the vectorizer - a list with one dictionary, because the transformer expects a list of records - and take the churn probability from the second column of predict_proba (index 1 is the probability of the positive class, "churn").
* We apply a threshold of 0.5 to get a yes/no answer, and cast NumPy values to native Python float and bool. JSON has no NumPy types, and Flask cannot serialize them.
* jsonify(result) turns the dictionary into a JSON response.


## Testing the service

* Run the service with "python predict.py": a browser can't test it - browsers send GET requests, and this route expects POST. 
* We use a small client script, "predict-test.py", which sends a request with the requests library.

* The customer dictionary holds the same fields the model saw during training. The json=customer argument serializes the dictionary to JSON in the request body. 
* Running the script prints:
{'churn': False, 'churn_probability': 0.3257561103397851}
not sending promo email to xyz-123

* This is exactly how a marketing service would talk to our model: send customer data as JSON, get the churn decision back, act on it.

## Running in production: gunicorn

When you start the app with python predict.py, Flask warns that its development server is not suitable for production - it is single-threaded and not built for heavy load.
* A WSGI server is the production replacement. WSGI (Web Server Gateway Interface) is the standard way Python web applications are served: the WSGI server imports our "app" object and handles all the network work around it.
* The common choice is gunicorn: gunicorn --bind=0.0.0.0:9696 predict:app
* The last argument is module:app_object - our file is predict.py and the Flask application inside it is called app. The service behaves the same, but now it is a real production server.

jiodspfosfpeof